In [2]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

initialDB_path = Path('../data/expandedDB.csv')
initialDB = pd.read_csv(initialDB_path, sep=',', header=0, comment='#', names=[
    'gene_name', 'gene_id', 'transcript_id', 'protein_id', "5' UTR", 'CDS', "3' UTR", 'protein_sequence', '5\' UTR_len', 'CDS_len', '3\' UTR_len', '5_UTR_GC', 'CDS_GC', '3_UTR_GC', 'global_GC', '5_AREs', '3_AREs', 'uAUG_count', 'uORF_in_frame', 'dORF_with_stop', 'dORF_truncated', '5_UTR', '3_UTR', '5_UTR_len', '3_UTR_len', 'IRES count', 'IRES IDs', '5_len', '3_len', '5_UTR_MFE', 'CDS_MFE', '3_UTR_MFE'
    ])
gtex_path = Path('../data/gtex_data.csv')
gtex_df = pd.read_csv(gtex_path, sep=',', header=0, comment='#', names=[
    'Name', 'Description', 'Adipose_Subcutaneous', 'Adipose_Visceral_Omentum', 'Adrenal_Gland', 
    'Artery_Aorta', 'Artery_Coronary', 'Artery_Tibial', 'Bladder', 'Brain_Amygdala', 'Brain_Anterior_cingulate_cortex_BA24', 
    'Brain_Caudate_basal_ganglia', 'Brain_Cerebellar_Hemisphere', 'Brain_Cerebellum', 'Brain_Cortex', 
    'Brain_Frontal_Cortex_BA9', 'Brain_Hippocampus', 'Brain_Hypothalamus', 'Brain_Nucleus_accumbens_basal_ganglia', 
    'Brain_Putamen_basal_ganglia', 'Brain_Spinal_cord_cervical_c-1', 'Brain_Substantia_nigra', 'Breast_Mammary_Tissue', 
    'Cells_Cultured_fibroblasts', 'Cells_EBV-transformed_lymphocytes', 'Cervix_Ectocervix', 'Cervix_Endocervix', 'Colon_Sigmoid', 
    'Colon_Transverse', 'Colon_Transverse_Mixed_Cell', 'Colon_Transverse_Mucosa', 'Colon_Transverse_Muscularis', 
    'Esophagus_Gastroesophageal_Junction', 'Esophagus_Mucosa', 'Esophagus_Muscularis', 'Fallopian_Tube', 'Heart_Atrial_Appendage', 
    'Heart_Left_Ventricle', 'Kidney_Cortex', 'Kidney_Medulla', 'Liver', 'Liver_Hepatocyte', 'Liver_Mixed_Cell', 'Liver_Portal_Tract', 
    'Lung', 'Minor_Salivary_Gland', 'Muscle_Skeletal', 'Nerve_Tibial', 'Ovary', 'Pancreas', 'Pancreas_Acini', 'Pancreas_Islets', 'Pancreas_Mixed_Cell',
    'Pituitary','Prostate','Skin_Not_Sun_Exposed_Suprapubic', 'Skin_Sun_Exposed_Lower_leg','Small_Intestine_Terminal_Ileum',
    'Small_Intestine_Terminal_Ileum_Lymphoid_Aggregate','Small_Intestine_Terminal_Ileum_Mixed_Cell','Spleen','Stomach',
    'Stomach_Mixed_Cell','Stomach_Mucosa','Stomach_Muscularis','Testis','Thyroid','Uterus','Vagina','Whole_Blood'])

In [3]:
gtex_df['clean_gene_id'] = gtex_df['Name'].astype('string').str.split('.').str[0]
initialDB['clean_gene_id'] = initialDB['gene_id'].astype('string').str.split('.').str[0]

tissue_cols = [col for col in gtex_df.columns if col not in ['Name', 'Description', 'clean_gene_id']]

def calculate_tau(row):
    """Calculates the Tau score (0 to 1) for a gene across all tissues."""
    expression_values = row[tissue_cols].values.astype(float)
    
    max_expression = np.max(expression_values)
    if max_expression == 0:
        return np.nan
    
    normalized_values = expression_values / max_expression
    
    N = len(tissue_cols)
    tau = np.sum(1 - normalized_values) / (N - 1)
    return tau

print("Calculating tissue specificity (Tau scores)...")
gtex_df['tau_score'] = gtex_df.apply(calculate_tau, axis=1)

gtex_df['primary_tissue'] = gtex_df[tissue_cols].idxmax(axis=1)
gtex_df['max_tpm'] = gtex_df[tissue_cols].max(axis=1)

def categorize_gene(row):
    if row['tau_score'] < 0.25 and row['max_tpm'] > 1:
        return "Housekeeping"
    elif row['tau_score'] > 0.80:
        return "Tissue_Specific"
    else:
        return "Intermediate"

gtex_df['gene_category'] = gtex_df.apply(categorize_gene, axis=1)

# 8. Filter columns to save space and merge with your UTR dataset
gtex_summary = gtex_df[['clean_gene_id', 'tau_score', 'primary_tissue', 'gene_category']]

# Left join keeps all your sequence data and attaches the matching GTEx scores
df = pd.merge(
    initialDB, 
    gtex_summary, 
    left_on="clean_gene_id", 
    right_on="clean_gene_id", 
    how="left"
)

print("Done!")


Calculating tissue specificity (Tau scores)...
Done!


In [4]:
tissue_summary_path = Path('../data/tissue_summary.csv')
tissue_summary = pd.read_csv(tissue_summary_path, sep=',', header=0, comment='#', names=[
    'Gene', 'Tissue', 'nTPM', 'Protein_Score'
])

tissue_summary.head()

,Gene,Tissue,nTPM,Protein_Score
0,ENSG00000000003,adipose tissue,20.1,0.0
1,ENSG00000000003,adrenal gland,13.3,0.0
2,ENSG00000000003,appendix,4.6,2.0
3,ENSG00000000003,bone marrow,0.6,0.0
4,ENSG00000000003,breast,24.0,3.0


In [5]:
df = pd.merge(
    df,
    tissue_summary.rename(columns={'Gene': 'clean_gene_id'}),
    on='clean_gene_id',
    how='left'
)

df.head()

,gene_name,gene_id,transcript_id,protein_id,5' UTR,CDS,3' UTR,protein_sequence,5' UTR_len,CDS_len,...,5_UTR_MFE,CDS_MFE,3_UTR_MFE,clean_gene_id,tau_score,primary_tissue,gene_category,Tissue,nTPM,Protein_Score
0,OR4F5,ENSG00000186092.7,ENST00000641515.2,ENSP00000493376.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,MKKVTAEAISWNESTSETNNSMVTEFIFLGLSDSQELQTFLFMLFF...,60,981,...,NaN,NaN,NaN,ENSG00000186092,0.867115,Colon_Transverse_Mucosa,Tissue_Specific,NaN,NaN,NaN
1,SAMD11,ENSG00000187634.14,ENST00000616016.5,ENSP00000478421.2,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2535,...,NaN,NaN,NaN,ENSG00000187634,0.861938,Pituitary,Tissue_Specific,NaN,NaN,NaN
2,NOC2L,ENSG00000188976.12,ENST00000327044.7,ENSP00000317992.6,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,16,2250,...,NaN,NaN,NaN,ENSG00000188976,0.516676,Cells_EBV-transformed_lymphocytes,Intermediate,adipose tissue,25.5,0.0
3,NOC2L,ENSG00000188976.12,ENST00000327044.7,ENSP00000317992.6,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,16,2250,...,NaN,NaN,NaN,ENSG00000188976,0.516676,Cells_EBV-transformed_lymphocytes,Intermediate,adrenal gland,22.5,0.0
4,NOC2L,ENSG00000188976.12,ENST00000327044.7,ENSP00000317992.6,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,16,2250,...,NaN,NaN,NaN,ENSG00000188976,0.516676,Cells_EBV-transformed_lymphocytes,Intermediate,appendix,21.7,0.0


In [6]:
df.drop(columns=['clean_gene_id'], inplace=True)

In [7]:
# IUPAC Kozak Sequence Identification

# •	Uppercase exact match (e.g.A→A):+1.0; lowercase:+0.5
# •	Degenerate (R/Y/W/S/M/K):+0.5; lowercase:+0.25
# •	N(any):+0.25; lowercase:+0.125
# •	No match:0
# •	Out-of-bounds positions are padded withN(score 0)
# •	PPM/PWM scoring: simply looks up matrix[pos][nucleotide] and sums.
# •	and the IUPAC sequence I used: GCCRCCatgG

def iupac_score(seq, pos):
    iupac_dict = {
        'A': {'A': 1.0, 'R': 0.5, 'Y': 0.0, 'W': 0.5, 'S': 0.0, 'M': 0.5, 'K': 0.0, 'N': 0.25},
        'C': {'C': 1.0, 'R': 0.0, 'Y': 0.5, 'W': 0.0, 'S': 0.5, 'M': 0.5, 'K': 0.0, 'N': 0.25},
        'G': {'G': 1.0, 'R': 0.5, 'Y': 0.0, 'W': 0.5, 'S': 0.5, 'M': 0.0, 'K': 0.5, 'N': 0.25},
        'T': {'T': 1.0, 'R': 0.0, 'Y': 0.5, 'W': 0.5, 'S': 0.0, 'M': 0.5, 'K': 0.5, 'N': 0.25},
        'a': {'A': 0.5, 'R': 0.25, 'Y': 0.0, 'W': 0.25, 'S': 0.0, 'M': 0.25, 'K': 0.0, 'N': 0.125},
        'c': {'C': 0.5, 'R': 0.0, 'Y': 0.25, 'W': 0.0, 'S': 0.25, 'M': 0.25, 'K': 0.0, 'N': 0.125},
        'g': {'G': 0.5, 'R': 0.25, 'Y': 0.0, 'W': 0.25, 'S': 0.25, 'M': 0.0, 'K': 0.25, 'N': 0.125},  
        't': {'T': 0.5, 'R': 0.0, 'Y': 0.25, 'W': 0.25, 'S': 0.0, 'M': 0.25, 'K': 0.25, 'N': 0.125},
    }
    iupac_seq = "GCCRCCatgG"
    if pos < 0 or pos >= len(iupac_seq):
        return 0.0
    iupac_char = iupac_seq[pos]
    return iupac_dict.get(iupac_char, {}).get(seq[pos].upper(), 0.0)


def kozak_score_row(row):
    iupac_len = len("GCCRCCatgG")
    utr = row.get('5_UTR') or row.get("5' UTR") or ''
    cds = row.get('CDS') or ''
    utr = str(utr)
    cds = str(cds)
    cds_prefix = cds[:4]  # atgG should come from CDS start
    utr_needed = iupac_len - len(cds_prefix)
    if utr_needed < 0:
        cds_prefix = cds_prefix[:4]
        utr_needed = iupac_len - len(cds_prefix)
    utr_suffix = utr[-utr_needed:] if utr_needed > 0 else ''
    utr_part = utr_suffix.rjust(utr_needed, 'N')
    cds_part = cds_prefix.ljust(iupac_len - utr_needed, 'N')
    combined = (utr_part + cds_part)[:iupac_len]
    return sum(iupac_score(combined, pos) for pos in range(iupac_len))

df['kozak_iupac_score'] = df.apply(kozak_score_row, axis=1)
df.head()

,gene_name,gene_id,transcript_id,protein_id,5' UTR,CDS,3' UTR,protein_sequence,5' UTR_len,CDS_len,...,5_UTR_MFE,CDS_MFE,3_UTR_MFE,tau_score,primary_tissue,gene_category,Tissue,nTPM,Protein_Score,kozak_iupac_score
0,OR4F5,ENSG00000186092.7,ENST00000641515.2,ENSP00000493376.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,MKKVTAEAISWNESTSETNNSMVTEFIFLGLSDSQELQTFLFMLFF...,60,981,...,NaN,NaN,NaN,0.867115,Colon_Transverse_Mucosa,Tissue_Specific,NaN,NaN,NaN,2.5
1,SAMD11,ENSG00000187634.14,ENST00000616016.5,ENSP00000478421.2,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2535,...,NaN,NaN,NaN,0.861938,Pituitary,Tissue_Specific,NaN,NaN,NaN,5.5
2,NOC2L,ENSG00000188976.12,ENST00000327044.7,ENSP00000317992.6,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,16,2250,...,NaN,NaN,NaN,0.516676,Cells_EBV-transformed_lymphocytes,Intermediate,adipose tissue,25.5,0.0,4.5
3,NOC2L,ENSG00000188976.12,ENST00000327044.7,ENSP00000317992.6,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,16,2250,...,NaN,NaN,NaN,0.516676,Cells_EBV-transformed_lymphocytes,Intermediate,adrenal gland,22.5,0.0,4.5
4,NOC2L,ENSG00000188976.12,ENST00000327044.7,ENSP00000317992.6,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,16,2250,...,NaN,NaN,NaN,0.516676,Cells_EBV-transformed_lymphocytes,Intermediate,appendix,21.7,0.0,4.5


In [8]:
def wobble_freq(seq, nt):
    if not isinstance(seq, str):
        return np.nan
    seq = seq.strip().upper()
    
    valid_len = (len(seq) // 3) * 3
    seq = seq[:valid_len]
    
    if not seq:
        return np.nan
        
    wobble_positions = seq[2::3]
    
    count = wobble_positions.count(nt)
    return count / len(wobble_positions)

df = initialDB.copy()
df['cds_wobble_nt_freq_A'] = df['CDS'].fillna('').map(lambda x: wobble_freq(x, 'A'))
df['cds_wobble_nt_freq_C'] = df['CDS'].fillna('').map(lambda x: wobble_freq(x, 'C'))
df['cds_wobble_nt_freq_G'] = df['CDS'].fillna('').map(lambda x: wobble_freq(x, 'G'))
df['cds_wobble_nt_freq_T'] = df['CDS'].fillna('').map(lambda x: wobble_freq(x, 'T'))


In [9]:
def calculate_true_enc(seq):
    # Standard genetic code mapping
    codon_to_aa = {
        'TTT':'F', 'TTC':'F', 'TTA':'L', 'TTG':'L', 'TCT':'S', 'TCC':'S', 'TCA':'S', 'TCG':'S',
        'TAT':'Y', 'TAC':'Y', 'TAA':'STOP', 'TAG':'STOP', 'TGT':'C', 'TGC':'C', 'TGA':'STOP', 'TGG':'W',
        'CTT':'L', 'CTC':'L', 'CTA':'L', 'CTG':'L', 'CCT':'P', 'CCC':'P', 'CCA':'P', 'CCG':'P',
        'CAT':'H', 'CAC':'H', 'CAA':'Q', 'CAG':'Q', 'CGT':'R', 'CGC':'R', 'CGA':'R', 'CGG':'R',
        'ATT':'I', 'ATC':'I', 'ATA':'I', 'ATG':'M', 'ACT':'T', 'ACC':'T', 'ACA':'T', 'ACG':'T',
        'AAT':'N', 'AAC':'N', 'AAA':'K', 'AAG':'K', 'AGT':'S', 'AGC':'S', 'AGA':'R', 'AGG':'R',
        'GTT':'V', 'GTC':'V', 'GTA':'V', 'GTG':'V', 'GCT':'A', 'GCC':'A', 'GCA':'A', 'GCG':'A',
        'GAT':'D', 'GAC':'D', 'GAA':'E', 'GAG':'E', 'GGT':'G', 'GGC':'G', 'GGA':'G', 'GGG':'G'
    }
    
    codons = [seq[i:i+3] for i in range(0, (len(seq)//3)*3, 3)]
    
    # 1. Group codons by Amino Acid
    aa_groups = {}
    for c in codons:
        aa = codon_to_aa.get(c)
        if aa and aa != 'STOP':
            aa_groups.setdefault(aa, []).append(c)
            
    # 2. Calculate Fi for each AA
    fi_list = []
    for aa, codon_list in aa_groups.items():
        if len(codon_list) <= 1: continue
        
        counts = pd.Series(codon_list).value_counts()
        n = len(codon_list)
        # F = sum(pi^2) where pi = count_i / n
        pi = counts / n
        fi = np.sum(pi**2)
        
        # Calculate Fi (weighted)
        if fi > 0:
            fi_list.append((fi * n - 1) / (n - 1))

    # 3. Final ENC
    # Weighting the Fi values based on the number of codons per AA
    return 2 + 9 / np.mean(fi_list) if fi_list else 61.0

# Apply
df['cds_enc'] = df['CDS'].fillna('').map(calculate_true_enc)

In [10]:
bases = ['A', 'C', 'G', 'T']
all_codons = [b1+b2+b3 for b1 in bases for b2 in bases for b3 in bases]

def get_codon_frequencies(seq):
    if not isinstance(seq, str) or len(seq) < 3:
        return pd.Series({f'cds_codon_freq_{c}': 0.0 for c in all_codons})
    
    seq = seq.upper()
    # Extract codons
    codons = [seq[i:i+3] for i in range(0, (len(seq)//3)*3, 3)]
    total_codons = len(codons)
    
    # Count occurrences
    counts = pd.Series(codons).value_counts()
    
    # Calculate frequencies
    freqs = {f'cds_codon_freq_{c}': counts.get(c, 0) / total_codons for c in all_codons}
    return pd.Series(freqs)

# Apply to dataframe
codon_freq_df = df['CDS'].fillna('').apply(get_codon_frequencies)
df = pd.concat([df, codon_freq_df], axis=1)

In [11]:
def update_stats_exact(df, col_name, prefix):
    # Check if column exists
    if col_name not in df.columns:
        print(f"[ERROR] Column '{col_name}' does not exist!")
        return

    # Convert to uppercase and force string type
    series = df[col_name].astype(str).str.upper()
    
    # Calculate lengths (replace 0 with NaN to avoid division by zero)
    lengths = series.str.len().replace(0, np.nan)
    
    # Calculate frequencies using exact column names
    df[f'{prefix}_nt_freq_A'] = series.str.count('A') / lengths
    df[f'{prefix}_nt_freq_C'] = series.str.count('C') / lengths
    df[f'{prefix}_nt_freq_G'] = series.str.count('G') / lengths
    df[f'{prefix}_nt_freq_T'] = series.str.count('T') / lengths
    
    # Fill remaining NaNs with 0
    cols = [f'{prefix}_nt_freq_A', f'{prefix}_nt_freq_C', f'{prefix}_nt_freq_G', f'{prefix}_nt_freq_T']
    df[cols] = df[cols].fillna(0.0)
    
    print(f"[SUCCESS] Processed {col_name} -> {prefix}")

update_stats_exact(df, "5' UTR", 'utr5')
update_stats_exact(df, "3' UTR", 'utr3')
update_stats_exact(df, 'CDS', 'cds')


[SUCCESS] Processed 5' UTR -> utr5
[SUCCESS] Processed 3' UTR -> utr3
[SUCCESS] Processed CDS -> cds


In [12]:
codon_cols = [c for c in df.columns if c.startswith('cds_codon_freq_') and len(c.split('_')[-1]) == 3]
freqs = df[codon_cols].mean() # Average frequency across all genes
threshold = freqs.quantile(0.10)
rare_codons = freqs[freqs < threshold].index.str.replace('cds_codon_freq_', '')

def calculate_rare_codon_stats(seq):
    if not isinstance(seq, str) or len(seq) < 3: return 0, ""
    codons = [seq[i:i+3] for i in range(0, (len(seq)//3)*3, 3)]
    
    # Identify positions (1-based)
    rare_mask = [c in rare_codons.values for c in codons]
    positions = [i+1 for i, is_rare in enumerate(rare_mask) if is_rare]
    
    # Identify clusters (consecutive)
    clusters = 0
    if len(positions) > 1:
        in_cluster = False
        for i in range(len(positions) - 1):
            if positions[i+1] == positions[i] + 1: # Consecutive
                if not in_cluster:
                    clusters += 1
                    in_cluster = True
            else:
                in_cluster = False
    return clusters, str(positions)

# Apply
res = df['CDS'].fillna('').apply(calculate_rare_codon_stats)
df['cds_rare_codon_n_clusters'] = [r[0] for r in res]
df['cds_rare_codon_positions'] = [r[1] for r in res]

In [13]:
import pandas as pd
tRNA = pd.read_csv('../data/hg38-tRNAs-confidence-set.out', sep='\s+', skiprows=3, header=None)

print(tRNA.head())

print(tRNA[[4, 5]].head())

# 3. Create a clean DataFrame with the right names
tRNA_clean = tRNA[[4, 5]].copy()
tRNA_clean.columns = ['Isotype', 'Anticodon']

# 4. Now count them
tRNA_counts = tRNA_clean['Anticodon'].value_counts()
# print(tRNA_counts)


     0   1          2          3    4    5         6         7     8     9   \
0  chr1   4   16861921   16861991  Gly  CCC         0         0  78.3  56.6   
1  chr1   5   16872583   16872654  Glu  TTC         0         0  71.6  44.7   
2  chr1   7   16889677   16889750  Asn  GTT         0         0  77.7  51.9   
3  chr1   9   93847573   93847657  Arg  TCT  93847610  93847621  71.1  51.2   
4  chr1  12  143691474  143691545  Gln  CTG         0         0  71.9  44.4   

     10   11   12     13         14    15          16   17  
0  21.7  inf  Gly  109.7  cytosolic  high  confidence  set  
1  26.9  inf  Glu  113.5  cytosolic  high  confidence  set  
2  25.8  inf  Asn  117.7  cytosolic  high  confidence  set  
3  19.9  inf  Arg  103.0  cytosolic  high  confidence  set  
4  27.5  inf  Gln  106.4  cytosolic  high  confidence  set  
     4    5
0  Gly  CCC
1  Glu  TTC
2  Asn  GTT
3  Arg  TCT
4  Gln  CTG


In [14]:
anticodon_counts = {
    'GCA': 29, 'AGC': 26, 'GTT': 25, 'CAT': 20, 'CTT': 15, 'AAT': 15,
    'GCC': 14, 'CTG': 13, 'CAC': 13, 'GTC': 13, 'GTA': 13, 'TTT': 12,
    'GAA': 10, 'AAC': 10, 'TCC': 9, 'GTG': 9, 'CAG': 9, 'AGG': 9, 
    'AAG': 9, 'AGT': 9, 'AGA': 9, 'TTC': 8, 'CTC': 8, 'GCT': 8, 
    'TGC': 8, 'CAA': 7, 'TGG': 7, 'CCA': 7, 'ACG': 7, 'TCT': 6, 
    'TGT': 6, 'TCG': 6, 'TTG': 6, 'CCC': 5, 'TAC': 5, 'CCT': 5, 
    'CGT': 5, 'TAT': 5, 'CGG': 4, 'TGA': 4, 'TAA': 4, 'CGA': 4, 
    'CCG': 4, 'CGC': 4, 'TAG': 3, 'GAT': 3, 'TCA': 1, 'ATA': 1
}

def get_wobble_pairs(anticodon):
    return anticodon[::-1].replace('T', 'U')

# 3. Create the w_i table
# https://academic.oup.com/nar/article/32/17/5036/2383185
w_values = {codon: count / max(anticodon_counts.values()) for codon, count in anticodon_counts.items()}

print(w_values)

{'GCA': 1.0, 'AGC': 0.896551724137931, 'GTT': 0.8620689655172413, 'CAT': 0.6896551724137931, 'CTT': 0.5172413793103449, 'AAT': 0.5172413793103449, 'GCC': 0.4827586206896552, 'CTG': 0.4482758620689655, 'CAC': 0.4482758620689655, 'GTC': 0.4482758620689655, 'GTA': 0.4482758620689655, 'TTT': 0.41379310344827586, 'GAA': 0.3448275862068966, 'AAC': 0.3448275862068966, 'TCC': 0.3103448275862069, 'GTG': 0.3103448275862069, 'CAG': 0.3103448275862069, 'AGG': 0.3103448275862069, 'AAG': 0.3103448275862069, 'AGT': 0.3103448275862069, 'AGA': 0.3103448275862069, 'TTC': 0.27586206896551724, 'CTC': 0.27586206896551724, 'GCT': 0.27586206896551724, 'TGC': 0.27586206896551724, 'CAA': 0.2413793103448276, 'TGG': 0.2413793103448276, 'CCA': 0.2413793103448276, 'ACG': 0.2413793103448276, 'TCT': 0.20689655172413793, 'TGT': 0.20689655172413793, 'TCG': 0.20689655172413793, 'TTG': 0.20689655172413793, 'CCC': 0.1724137931034483, 'TAC': 0.1724137931034483, 'CCT': 0.1724137931034483, 'CGT': 0.1724137931034483, 'TAT': 

In [15]:
df.head()

,gene_name,gene_id,transcript_id,protein_id,5' UTR,CDS,3' UTR,protein_sequence,5' UTR_len,CDS_len,...,utr3_nt_freq_A,utr3_nt_freq_C,utr3_nt_freq_G,utr3_nt_freq_T,cds_nt_freq_A,cds_nt_freq_C,cds_nt_freq_G,cds_nt_freq_T,cds_rare_codon_n_clusters,cds_rare_codon_positions
0,OR4F5,ENSG00000186092.7,ENST00000641515.2,ENSP00000493376.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,MKKVTAEAISWNESTSETNNSMVTEFIFLGLSDSQELQTFLFMLFF...,60,981,...,0.000000,0.000000,0.000000,0.000000,0.240571,0.232416,0.193680,0.333333,0,"[15, 18, 170, 247, 310, 327]"
1,SAMD11,ENSG00000187634.14,ENST00000616016.5,ENSP00000478421.2,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2535,...,0.000000,0.000000,0.000000,0.000000,0.159763,0.373176,0.327811,0.139250,1,"[20, 51, 55, 96, 149, 161, 178, 198, 226, 259,..."
2,NOC2L,ENSG00000188976.12,ENST00000327044.7,ENSP00000317992.6,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,MAAAGSRKRRLAELTVDEFLASGFDSESESESENSPQAETREAREA...,16,2250,...,0.186235,0.275304,0.297571,0.240891,0.218667,0.284000,0.319111,0.178222,1,"[15, 22, 58, 62, 65, 100, 178, 185, 199, 247, ..."
3,KLHL17,ENSG00000187961.16,ENST00000338591.8,ENSP00000343930.3,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,ATGCAGCCCCGCAGCGAGCGCCCGGCCGGCAGGACGCAGAGCCCGG...,NaN,MQPRSERPAGRTQSPEHGSPGPGPEAPPPPPPQPPAPEAERTRPRQ...,110,1929,...,0.000000,0.000000,0.000000,0.000000,0.172110,0.330223,0.332297,0.165371,0,"[12, 42, 104, 134, 152, 178, 257, 333, 364, 37..."
4,PLEKHN1,ENSG00000187583.12,ENST00000379410.8,ENSP00000368720.3,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,ATGGGGAACAGCCACTGTGTCCCTCAGGCCCCCAGGAGGCTCCGGG...,NaN,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...,50,1836,...,0.000000,0.000000,0.000000,0.000000,0.174837,0.363290,0.303922,0.157952,0,"[23, 35, 45, 61, 70, 131, 143, 154, 189, 200, ..."


In [16]:
df.to_csv('../data/finalDB.csv', index=False)